This program has following functions
1. get_data(type, city, days, alerts) - program generates url based on these parameters and  fetches data for current weather, forecasted weather and alert data
2. get_city() - user has input to type Name of the City
3. get_days() - input number of days for which forecast data is required
4. Then 3 functions extract data from three json objects
    extract_current_data - data from current json
    extract_forecast_data - data from forecast json
    extract_alert_data - data from alert json
5. Output looks like this

City Chicago Current temp 70.0 F and Condition is Partly cloudy
Forecast for the next 15 days is
  Date      Min    Max  Sunrise  Sunset     Condition  
2026-04-24 55.8F 68.2F 05:57 AM 07:42 PM   Patchy rain nearby 
...
Your city has active alerts
Alert number 1
identifier - 4a66ed85-3fa7-11f1-8583-b63eaaffe11e
headline - Flood Warning issued April 23 at 9:10PM CDT until April 25 at 7:00AM CDT by NWS Chicago IL    

In [200]:
import requests
import pandas as pd 
from datetime import datetime

baseurl = "http://api.weatherapi.com/v1"
key = "f67eb958b8fd48839d2190431262104"
city = "Louisville"
type = "current"

In [201]:
def get_data(type, city, days, alerts):
    geturl = f"{baseurl}/{type}.json?key={key}&q={city}"
    if days != 0:
        geturl += f"&days={days}"
    if alerts == 'yes':
        geturl += f"&alerts={alerts}"
    response = requests.get(geturl)
    return response.status_code, response.json()

In [202]:
def get_city():
    city = input("Type the name of the city you want the weather data : ")
    return city

In [203]:
def get_days():
    days = int(input("Type the number of days for forecast data : "))
    return days

In [204]:
def extract_current_data(data):
    curr_temp = data['current']['temp_f']
    curr_cond = data['current']['condition']['text']
    return curr_temp, curr_cond

In [205]:
def extract_forecast_data(data):
    fore_list=[]
    for days in data['forecast']['forecastday']:
        for dayi,dayv in days.items():
            if dayi == 'date':
                date = dayv
            if dayi == 'day':
                for ji, jv in dayv.items():
                    if ji == 'maxtemp_f':
                        max_temp = jv
                    if ji == 'mintemp_f':
                        min_temp = jv
                    if ji == 'condition':
                        cond = jv['text']
            if dayi == 'astro':
                sunrise = dayv['sunrise']
                sunset = dayv['sunset']

        fore_dict = {}
        fore_dict['date']=date
        fore_dict['max_temp']=max_temp
        fore_dict['min_temp']=min_temp
        fore_dict['cond']=cond
        fore_dict['sunrise']=sunrise
        fore_dict['sunset']=sunset
        fore_list.append(fore_dict)
    return fore_list

In [206]:
def extract_alert_data(al_data):
    active_alerts = al_data['alerts']
    return active_alerts

In [207]:
city = get_city()
type = "current"
days = 0
alerts = ''
curr_status, curr_data = get_data(type, city,days,alerts)
if curr_status == 200:
    curr_temp, curr_cond = extract_current_data(curr_data)

    type = "forecast"
    foredays = get_days()
    fc_status, fc_data = get_data(type, city,foredays,alerts)
    fore_list= extract_forecast_data(fc_data)

    type = "alerts"
    alerts = 'yes'
    al_status, al_data = get_data(type, city,days,alerts)
    active_alerts = extract_alert_data(al_data)
else:
    print (f"Something went wrong, we hit error {curr_status}")

In [209]:
print ("\n")
print (f"City {city} Current temp {curr_temp} F and Condition is {curr_cond}")
print ("\n")
print (f"Forecast for the next {foredays} days is")
print (f"  Date      Min    Max  Sunrise  Sunset     Condition  ")
for i in fore_list:
    print (f"{i['date']} {i['min_temp']}F {i['max_temp']}F {i['sunrise']} {i['sunset']}   {i['cond']} ")
print ("\n")
if len(active_alerts['alert']) == 0:
    print ("No active alerts for your city")
else:
    counter = 0
    for i in active_alerts['alert']:
        counter +=1
        print ("Your city has active alerts")
        print (f"Alert number {counter}")
        for items, val in i.items():
            print (f"{items} - {val}")
        print ("\n")



City Chicago Current temp 70.0 F and Condition is Partly cloudy


Forecast for the next 15 days is
  Date      Min    Max  Sunrise  Sunset     Condition  
2026-04-24 55.8F 68.2F 05:57 AM 07:42 PM   Patchy rain nearby 
2026-04-25 44.9F 52.7F 05:55 AM 07:43 PM   Sunny 
2026-04-26 45.3F 59.4F 05:54 AM 07:44 PM   Sunny 
2026-04-27 51.9F 63.8F 05:52 AM 07:45 PM   Heavy rain 
2026-04-28 51.4F 61.4F 05:51 AM 07:46 PM   Patchy rain nearby 
2026-04-29 49.1F 58.2F 05:50 AM 07:47 PM   Sunny 
2026-04-30 47.5F 50.3F 05:48 AM 07:48 PM   Patchy rain nearby 
2026-05-01 45.7F 55.9F 05:47 AM 07:50 PM   Sunny 
2026-05-02 46.6F 59.6F 05:46 AM 07:51 PM   Sunny 
2026-05-03 50.8F 62.4F 05:44 AM 07:52 PM   Sunny 
2026-05-04 46.7F 54.0F 05:43 AM 07:53 PM   Partly Cloudy 
2026-05-05 47.2F 66.2F 05:42 AM 07:54 PM   Partly Cloudy 
2026-05-06 41.2F 49.8F 05:41 AM 07:55 PM   Patchy rain nearby 
2026-05-07 44.1F 55.4F 05:39 AM 07:56 PM   Sunny 


Your city has active alerts
Alert number 1
identifier - 4a66ed85-3fa